In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!rm -rf /content/kaggle_data /content/tf_speech
!mkdir -p /content/kaggle_data /content/tf_speech

!kaggle competitions download \
  -c tensorflow-speech-recognition-challenge \
  -f train.7z \
  -p /content/kaggle_data \
  --force

100% 1.04G/1.04G [00:30<00:00, 37.1MB/s]



In [ ]:
!apt-get -qq install -y p7zip-full

!7z x "/content/kaggle_data/train.7z" "-o/content/tf_speech" -y


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,48 CPUs AMD EPYC 9B45 (B00F21),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/kaggle_data/                               1 file, 1121103842 bytes (1070 MiB)

Extracting archive: /content/kaggle_data/train.7z
--
Path = /content/kaggle_data/train.7z
Type = 7z
Physical Size = 1121103842
Headers Size = 389133
Method = Delta LZMA2:24
Solid = +
Blocks = 2

  0%      0% 40 - train/audio/_background_noise_/exercise_bike.wav                                                            0% 43 - train/audio/_background_noise_/white_noise.wav                                                        

In [ ]:
!pip install silero-vad

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 28.6 MB/s eta 0:00:00


In [ ]:
import torch

print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("device capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)
print("supported archs:", torch.cuda.get_arch_list() if torch.cuda.is_available() else None)

torch: 2.10.0+cu128
torch cuda: 12.8
cuda available: True
device name: NVIDIA RTX PRO 6000 Blackwell Server Edition
device capability: (12, 0)
supported archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


# dataset.py

In [ ]:
%%writefile dataset.py
import random

import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
import torchaudio.compliance.kaldi as kaldi
from torch.utils.data import Dataset


class SpeechCommandsDataset(Dataset):
    def __init__(self, file_paths, labels, config, noise_paths=None, is_train=True):
        self.file_paths = file_paths
        self.labels = labels
        self.config = config
        self.noise_paths = noise_paths
        self.is_train = is_train

        self.sample_rate = config.get("sample_rate", 16000)
        self.num_samples = int(self.sample_rate * config.get("clip_duration_sec", 1.0))
        self.model_type = config.get("model_type", "").upper()

        # AST paper: 128 log-Mel fbank, 25 ms window, 10 ms shift.
        self.ast_use_kaldi_fbank = (
            self.model_type == "AST"
            and config.get("preprocessing", "mel_spectrogram") == "mel_spectrogram"
            and config.get("ast_use_kaldi_fbank", True)
        )

        self.mel_transform = T.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=config.get("n_fft", 400),
            win_length=config.get("win_length", 400),
            hop_length=config.get("hop_length", 160),
            n_mels=config.get("input_size", 128),
            window_fn=torch.hamming_window,
            center=config.get("stft_center", True),
        )

        self.mfcc_transform = T.MFCC(
            sample_rate=self.sample_rate,
            n_mfcc=config.get("n_mfcc", 40),
            melkwargs={
                "n_mels": config.get("mfcc_n_mels", 128),
                "n_fft": config.get("n_fft", 400),
                "win_length": config.get("win_length", 400),
                "hop_length": config.get("hop_length", 160),
                "window_fn": torch.hamming_window,
                "center": config.get("stft_center", True),
            },
        )

        self.time_masking = T.TimeMasking(
            time_mask_param=config.get("time_mask_param", 8)
        )

        self.freq_masking = T.FrequencyMasking(
            freq_mask_param=config.get("freq_mask_param", 8)
        )

    def __len__(self):
        return len(self.file_paths)

    def _pad_or_crop(self, waveform):
        """Pad or crop waveform to exactly self.num_samples samples."""
        if waveform.shape[1] > self.num_samples:
            waveform = waveform[:, : self.num_samples]
        elif waveform.shape[1] < self.num_samples:
            padding = self.num_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        return waveform

    def _pad_or_crop_features(self, features, max_length):
        """Pad or crop feature tensor shaped as (T, F)."""
        if max_length is None:
            return features

        t_len = features.shape[0]

        if t_len > max_length:
            return features[:max_length, :]

        if t_len < max_length:
            pad = torch.zeros(
                max_length - t_len,
                features.shape[1],
                dtype=features.dtype,
                device=features.device,
            )
            return torch.cat([features, pad], dim=0)

        return features

    def _normalize_for_ast(self, features):
        if self.model_type != "AST":
            return features

        if not self.config.get("ast_normalize", True):
            return features

        mean = features.mean()
        std = features.std().clamp_min(1e-6)

        return (features - mean) / std * self.config.get("ast_target_std", 0.5)

    def _apply_spec_augment(self, features):
        """
        Apply SpecAugment to features shaped as (T, F).

        Works for both:
        - MFCC: (T, n_mfcc)
        - log-mel/fbank: (T, n_mels)
        """
        if not self.is_train:
            return features

        if self.config.get("augmentation", "none") != "spec_augment":
            return features

        if random.random() > self.config.get("specaugment_prob", 1.0):
            return features

        # torchaudio masking expects shape (..., freq, time)
        spec = features.transpose(0, 1).unsqueeze(0)  # (1, F, T)

        for _ in range(self.config.get("num_time_masks", 2)):
            spec = self.time_masking(spec)

        for _ in range(self.config.get("num_freq_masks", 1)):
            spec = self.freq_masking(spec)

        features = spec.squeeze(0).transpose(0, 1)  # (T, F)

        return features

    def _prepare_features_for_model(self, features):
        """
        Common postprocessing for all feature types.

        Input shape: (T, F)
        Output shape:
        - AST: fixed (max_length, F), normalized
        - other models: unchanged except optional SpecAugment
        """
        if self.model_type == "AST":
            features = self._pad_or_crop_features(
                features,
                self.config.get("max_length", 100),
            )

        features = self._apply_spec_augment(features)

        if self.model_type == "AST":
            features = self._normalize_for_ast(features)

        return features

    def _prepare_noise_segment(self, noise):
        """Convert noise to mono and crop/pad it to exactly self.num_samples samples."""
        if noise.shape[0] > 1:
            noise = noise.mean(dim=0, keepdim=True)

        if noise.shape[1] > self.num_samples:
            start = random.randint(0, noise.shape[1] - self.num_samples)
            noise = noise[:, start : start + self.num_samples]
        else:
            noise = self._pad_or_crop(noise)

        return noise

    def _add_background_noise(self, waveform):
        """
        Mix background noise into waveform.

        waveform and noise always have matching shapes before addition.
        """
        waveform = self._pad_or_crop(waveform)

        if self.noise_paths is None or len(self.noise_paths) == 0:
            return waveform

        p = self.config.get("background_noise_prob", 0.7)
        if random.random() > p:
            return waveform

        noise_file = random.choice(self.noise_paths)
        noise, sr = torchaudio.load(noise_file)

        if sr != self.sample_rate:
            noise = F.resample(noise, orig_freq=sr, new_freq=self.sample_rate)

        noise = self._prepare_noise_segment(noise)
        noise = noise.to(device=waveform.device, dtype=waveform.dtype)

        snr_db = random.uniform(
            self.config.get("background_snr_min_db", 5.0),
            self.config.get("background_snr_max_db", 20.0),
        )

        speech_power = waveform.norm(p=2)
        noise_power = noise.norm(p=2)

        if noise_power.item() == 0:
            return waveform

        snr = 10 ** (snr_db / 20.0)
        scale = speech_power / (snr * noise_power)

        mixed = waveform + scale * noise

        if self.config.get("clamp_waveform_after_noise", False):
            mixed = mixed.clamp(-1.0, 1.0)

        return mixed

    def _ast_fbank_features(self, waveform):
        features = kaldi.fbank(
            waveform,
            num_mel_bins=self.config.get("input_size", 128),
            sample_frequency=self.sample_rate,
            frame_length=25.0,
            frame_shift=10.0,
            dither=0.0,
            energy_floor=0.0,
            window_type="hamming",
            use_log_fbank=True,
            snip_edges=False,
        )
        return features  # (T, F)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.file_paths[idx])
        label = self.labels[idx]

        if sr != self.sample_rate:
            waveform = F.resample(
                waveform,
                orig_freq=sr,
                new_freq=self.sample_rate,
            )

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        aug_type = self.config.get("augmentation", "none")

        # --- waveform-domain augmentation, train only ---
        if self.is_train and aug_type == "speed_perturbation":
            speed_choices = self.config.get("speed_factors", [0.9, 1.0, 1.1])
            speed_factor = random.choice(speed_choices)

            if speed_factor != 1.0:
                new_sr = int(self.sample_rate * speed_factor)
                waveform = F.resample(
                    waveform,
                    orig_freq=self.sample_rate,
                    new_freq=new_sr,
                )

        waveform = self._pad_or_crop(waveform)

        if self.is_train and aug_type == "background_noise":
            waveform = self._add_background_noise(waveform)

        preprocessing = self.config["preprocessing"]

        # --- MFCC ---
        if preprocessing == "mfcc":
            features = self.mfcc_transform(waveform)
            features = features.squeeze(0).transpose(0, 1)  # (T, F)
            features = self._prepare_features_for_model(features)

        # --- log-mel / fbank ---
        elif preprocessing == "mel_spectrogram":
            if self.ast_use_kaldi_fbank:
                features = self._ast_fbank_features(waveform)  # (T, F)
            else:
                features = self.mel_transform(waveform)
                features = torch.log(features + 1e-6)
                features = features.squeeze(0).transpose(0, 1)  # (T, F)

            features = self._prepare_features_for_model(features)

        # --- raw waveform ---
        else:
            features = waveform.squeeze(0).unsqueeze(-1)

        return features, label

Writing dataset.py


# models.py

For mel-spectrogram

In [ ]:
# %%writefile models.py
# import torch
# import torch.nn as nn
# from transformers import ASTConfig, ASTModel


# class LSTMClassifier(nn.Module):
#     def __init__(self, input_size, num_classes, num_layers=2, dropout=0.1):
#         super().__init__()
#         lstm_dropout = dropout if num_layers > 1 else 0.0
#         self.lstm = nn.LSTM(
#             input_size,
#             hidden_size=128,
#             num_layers=num_layers,
#             dropout=lstm_dropout,
#             batch_first=True,
#         )
#         self.fc = nn.Linear(128, num_classes)

#     def forward(self, x):
#         out, _ = self.lstm(x)
#         out = out[:, -1, :]
#         return self.fc(out)


# class CRNNClassifier(nn.Module):
#     def __init__(
#         self,
#         input_size,
#         num_classes,
#         conv_filters=64,
#         gru_hidden=256,
#         dropout=0.1,
#         fc1=128,
#         fc2=256,
#     ):
#         super().__init__()
#         self.conv1 = nn.Conv2d(1, conv_filters, kernel_size=(3, 3), padding=(1, 1))
#         self.conv2 = nn.Conv2d(conv_filters, conv_filters, kernel_size=(5, 3), padding=(2, 1))
#         self.relu = nn.ReLU()

#         self.gru = nn.GRU(
#             input_size=input_size * conv_filters,
#             hidden_size=gru_hidden,
#             num_layers=1,
#             batch_first=True,
#         )

#         self.dropout = nn.Dropout(dropout)
#         self.fc1 = nn.Linear(gru_hidden, fc1)
#         self.fc2 = nn.Linear(fc1, fc2)
#         self.out = nn.Linear(fc2, num_classes)

#     def forward(self, x):
#         x = x.unsqueeze(1)                  # (B, 1, T, F)
#         x = self.relu(self.conv1(x))        # (B, C, T, F)
#         x = self.relu(self.conv2(x))        # (B, C, T, F)

#         x = x.permute(0, 2, 3, 1).contiguous()
#         B, T, F, C = x.shape
#         x = x.view(B, T, F * C)

#         x, _ = self.gru(x)
#         x = x[:, -1, :]

#         x = self.dropout(x)
#         x = self.relu(self.fc1(x))
#         x = self.relu(self.fc2(x))
#         return self.out(x)


# class ASTClassifier(nn.Module):
#     def __init__(
#         self,
#         input_size,
#         num_classes,
#         num_heads=12,
#         max_length=100,
#         hidden_size=384,
#         num_hidden_layers=6,
#         dropout=0.1,
#         patch_size=16,
#         frequency_stride=10,
#         time_stride=10,
#     ):
#         super().__init__()
#         checkpoint = "MIT/ast-finetuned-audioset-10-10-0.4593"

#         config = ASTConfig.from_pretrained(checkpoint)

#         assert hidden_size % num_heads == 0

#         config.num_mel_bins = input_size
#         config.max_length = max_length
#         config.hidden_size = hidden_size
#         config.num_hidden_layers = num_hidden_layers
#         config.num_attention_heads = num_heads
#         config.intermediate_size = hidden_size * 4
#         config.hidden_dropout_prob = dropout
#         config.attention_probs_dropout_prob = dropout
#         config.patch_size = patch_size
#         config.frequency_stride = frequency_stride
#         config.time_stride = time_stride

#         self.backbone = ASTModel.from_pretrained(
#             checkpoint,
#             config=config,
#             ignore_mismatched_sizes=True,
#         )

#         self.backbone.gradient_checkpointing_enable()

#         self.dropout = nn.Dropout(dropout)
#         self.fc = nn.Linear(hidden_size, num_classes)

#         print(
#             "AST position embeddings:",
#             self.backbone.embeddings.position_embeddings.shape
#         )

#     def forward(self, x):
#         # x musi mieć shape (B, T, F), np. (B, 100, 128)
#         outputs = self.backbone(input_values=x)
#         pooled = outputs.pooler_output
#         pooled = self.dropout(pooled)
#         return self.fc(pooled)


# class SileroVADClassifier:
#     def __init__(self):
#         from silero_vad import load_silero_vad, get_speech_timestamps
#         self.model = load_silero_vad()
#         self.get_speech_timestamps = get_speech_timestamps

#     @torch.no_grad()
#     def predict(self, wav, sampling_rate=16000):
#         if wav.dim() == 2:
#             wav = wav.squeeze(0)
#         if wav.dim() != 1:
#             raise ValueError("wav musi mieć shape (N,) albo (1, N)")

#         speech_timestamps = self.get_speech_timestamps(
#             wav,
#             self.model,
#             sampling_rate=sampling_rate,
#         )
#         return 1 if len(speech_timestamps) > 0 else 0

#     @torch.no_grad()
#     def predict_batch(self, wav_batch, sampling_rate=16000):
#         if isinstance(wav_batch, torch.Tensor):
#             return torch.tensor(
#                 [self.predict(w, sampling_rate) for w in wav_batch],
#                 dtype=torch.long,
#             )

#         return torch.tensor(
#             [self.predict(w, sampling_rate) for w in wav_batch],
#             dtype=torch.long,
#         )


For mfcc

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
from transformers import ASTConfig, ASTModel


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, num_classes, num_layers=2, dropout=0.1):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size,
            hidden_size=128,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)


class CRNNClassifier(nn.Module):
    def __init__(
        self,
        input_size,
        num_classes,
        conv_filters=64,
        gru_hidden=256,
        dropout=0.1,
        fc1=128,
        fc2=256,
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(1, conv_filters, kernel_size=(3, 3), padding=(1, 1))
        self.conv2 = nn.Conv2d(conv_filters, conv_filters, kernel_size=(5, 3), padding=(2, 1))
        self.relu = nn.ReLU()

        self.gru = nn.GRU(
            input_size=input_size * conv_filters,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(gru_hidden, fc1)
        self.fc2 = nn.Linear(fc1, fc2)
        self.out = nn.Linear(fc2, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)                  # (B, 1, T, F)
        x = self.relu(self.conv1(x))        # (B, C, T, F)
        x = self.relu(self.conv2(x))        # (B, C, T, F)

        x = x.permute(0, 2, 3, 1).contiguous()
        B, T, F, C = x.shape
        x = x.view(B, T, F * C)

        x, _ = self.gru(x)
        x = x[:, -1, :]

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.out(x)


class ASTClassifier(nn.Module):
    def __init__(
        self,
        input_size,
        num_classes,
        num_heads=12,
        max_length=100,
        hidden_size=768,
        num_hidden_layers=12,
        dropout=0.1,
        patch_size=16,
        frequency_stride=10,
        time_stride=10,
        use_gradient_checkpointing=True,
    ):
        super().__init__()

        checkpoint = "MIT/ast-finetuned-audioset-10-10-0.4593"

        assert hidden_size % num_heads == 0

        self.input_size = input_size
        self.max_length = max_length

        config = ASTConfig.from_pretrained(checkpoint)

        config.num_mel_bins = input_size
        config.max_length = max_length
        config.hidden_size = hidden_size
        config.num_hidden_layers = num_hidden_layers
        config.num_attention_heads = num_heads
        config.intermediate_size = hidden_size * 4
        config.hidden_dropout_prob = dropout
        config.attention_probs_dropout_prob = dropout
        config.patch_size = patch_size
        config.frequency_stride = frequency_stride
        config.time_stride = time_stride

        self.backbone = ASTModel.from_pretrained(
            checkpoint,
            config=config,
            ignore_mismatched_sizes=True,
        )

        if use_gradient_checkpointing:
            self.backbone.gradient_checkpointing_enable()

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

        print(
            "AST position embeddings:",
            self.backbone.embeddings.position_embeddings.shape
        )

    def forward(self, x):
        # x: (B, T, F), np. MFCC: (B, 100, 40), log-mel: (B, 100, 128)

        if x.dim() != 3:
            raise ValueError(f"AST expects x shape (B, T, F), got {x.shape}")

        if x.shape[1] != self.max_length:
            raise ValueError(
                f"Wrong time length. Expected T={self.max_length}, got {x.shape[1]}"
            )

        if x.shape[2] != self.input_size:
            raise ValueError(
                f"Wrong feature size. Expected F={self.input_size}, got {x.shape[2]}"
            )

        x = x.float()

        outputs = self.backbone(input_values=x)
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)

        return self.fc(pooled)


class SileroVADClassifier:
    def __init__(self):
        from silero_vad import load_silero_vad, get_speech_timestamps
        self.model = load_silero_vad()
        self.get_speech_timestamps = get_speech_timestamps

    @torch.no_grad()
    def predict(self, wav, sampling_rate=16000):
        if wav.dim() == 2:
            wav = wav.squeeze(0)
        if wav.dim() != 1:
            raise ValueError("wav musi mieć shape (N,) albo (1, N)")

        speech_timestamps = self.get_speech_timestamps(
            wav,
            self.model,
            sampling_rate=sampling_rate,
        )
        return 1 if len(speech_timestamps) > 0 else 0

    @torch.no_grad()
    def predict_batch(self, wav_batch, sampling_rate=16000):
        if isinstance(wav_batch, torch.Tensor):
            return torch.tensor(
                [self.predict(w, sampling_rate) for w in wav_batch],
                dtype=torch.long,
            )

        return torch.tensor(
            [self.predict(w, sampling_rate) for w in wav_batch],
            dtype=torch.long,
        )

Writing models.py


# engine.py

In [ ]:
%%writefile engine.py
import copy
import csv
import json
import os
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix
from tqdm import tqdm

from dataset import SpeechCommandsDataset
from models import LSTMClassifier, CRNNClassifier, ASTClassifier, SileroVADClassifier


def build_model(config):
    model_type = config["model_type"].upper()
    num_classes = config["num_classes"]

    if model_type == "LSTM":
        return LSTMClassifier(
            input_size=config["input_size"],
            num_classes=num_classes,
            num_layers=config.get("num_layers", 2),
            dropout=config.get("dropout", 0.1),
        )

    if model_type in ("CNN+GRU", "CRNN"):
        return CRNNClassifier(
            input_size=config["input_size"],
            num_classes=num_classes,
            conv_filters=config.get("conv_filters", 64),
            gru_hidden=config.get("gru_hidden", 128),
            dropout=config.get("dropout", 0.1),
        )

    if model_type == "AST":
        return ASTClassifier(
            input_size=config["input_size"],
            num_classes=num_classes,
            num_heads=config.get("num_heads", 12),
            max_length=config.get("max_length", 100),
            hidden_size=config.get("hidden_size", 768),
            num_hidden_layers=config.get("num_hidden_layers", 12),
            dropout=config.get("dropout", 0.1),
            patch_size=config.get("patch_size", 16),
            frequency_stride=config.get("frequency_stride", 10),
            time_stride=config.get("time_stride", 10),
        )

    raise ValueError(f"Nieznany model_type: {config['model_type']}")


def validate_experiment_config(config):
    approach = config["approach"]

    if approach not in ("two_stage", "single_stage"):
        raise ValueError("approach musi być 'two_stage' albo 'single_stage'")

    if approach == "two_stage" and config["num_classes"] != 11:
        raise ValueError(
            "Dla approach='two_stage' num_classes musi wynosić 11. "
            "Model etapu 2 uczy się tylko: 10 target_words + unknown."
        )

    if approach == "single_stage" and config["num_classes"] != 12:
        raise ValueError("Dla approach='single_stage' num_classes musi wynosić 12")


def build_scheduler(optimizer, config, num_epochs):
    if config.get("use_ast_paper_lr_decay", False):
        def lr_lambda(epoch):
            return 1.0 if epoch < 5 else 0.85 ** (epoch - 4)
        return LambdaLR(optimizer, lr_lambda=lr_lambda)

    if config.get("use_cosine_annealing", False):
        return CosineAnnealingLR(optimizer, T_max=num_epochs)

    return None


def build_optimizer(model, config):
    lr = config["learning_rate"]
    weight_decay = config.get("weight_decay", 0.0)
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


def build_criterion(config, device):
    class_weights = config.get("class_weights", None)
    if class_weights is not None:
        if len(class_weights) != config["num_classes"]:
            raise ValueError(
                f"class_weights ma długość {len(class_weights)}, "
                f"a num_classes={config['num_classes']}."
            )
        class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)
        return nn.CrossEntropyLoss(weight=class_weights)
    return nn.CrossEntropyLoss()


def _classification_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if len(y_true) == 0:
        return {
            f"{prefix}macro_f1": np.nan,
            f"{prefix}balanced_acc": np.nan,
            f"{prefix}overall_acc": np.nan,
            f"{prefix}count": 0,
        }

    return {
        f"{prefix}macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}balanced_acc": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}overall_acc": accuracy_score(y_true, y_pred),
        f"{prefix}count": int(len(y_true)),
    }


def flatten_dict(d, prefix=""):
    flat = {}
    for k, v in d.items():
        key = f"{prefix}{k}" if prefix == "" else f"{prefix}.{k}"
        if isinstance(v, dict):
            flat.update(flatten_dict(v, key))
        else:
            flat[key] = v
    return flat


def save_rows_csv(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    rows = list(rows)
    if not rows:
        return

    keys = []
    for row in rows:
        for key in row.keys():
            if key not in keys:
                keys.append(key)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def save_metrics_csv(metrics, path):
    flat = flatten_dict(metrics)
    save_rows_csv([flat], path)


def save_confusion_matrix_csv(y_true, y_pred, labels, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for i, true_label in enumerate(labels):
        row = {"true_label": true_label}
        for j, pred_label in enumerate(labels):
            row[f"pred_{pred_label}"] = int(cm[i, j])
        rows.append(row)
    save_rows_csv(rows, path)


def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)


@torch.no_grad()
def evaluate_model(model, dataloader, criterion, device, return_preds=False):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for inputs, labels in tqdm(dataloader, desc="Validation", leave=False):
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    metrics = {
        "loss": running_loss / max(len(dataloader), 1),
        **_classification_metrics(all_labels, all_preds),
    }

    if return_preds:
        return metrics, np.asarray(all_labels), np.asarray(all_preds)

    return metrics


def run_experiment(config, train_loader, val_loader, num_epochs=10, device="cuda"):
    validate_experiment_config(config)

    model = build_model(config).to(device)
    criterion = build_criterion(config, device)
    optimizer = build_optimizer(model, config)
    scheduler = build_scheduler(optimizer, config, num_epochs)

    best_val_f1 = -1.0
    best_metrics = None
    best_state = copy.deepcopy(model.state_dict())
    history = []

    save_dir = config.get("save_dir", "checkpoints")
    experiment_name = config.get(
        "experiment_name",
        f"{config['approach']}_{config['model_type']}_{config['preprocessing']}_{config.get('augmentation', 'none')}_{config.get('seed', 42)}",
    )
    experiment_dir = os.path.join(save_dir, experiment_name)
    os.makedirs(experiment_dir, exist_ok=True)

    save_json(config, os.path.join(experiment_dir, "config.json"))

    print("=" * 80)
    print(f"Experiment : {experiment_name}")
    print(f"Approach   : {config['approach']}")
    print(f"Model      : {config['model_type']}")
    print(f"Num classes: {config['num_classes']}")
    print(f"LR         : {config['learning_rate']}")
    print("=" * 80)

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate_model(model, val_loader, criterion, device)

        lr = optimizer.param_groups[0]["lr"]

        if scheduler is not None:
            scheduler.step()

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "lr": lr,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Macro F1: {val_metrics['macro_f1']:.4f} | "
            f"Val Balanced Acc: {val_metrics['balanced_acc']:.4f} | "
            f"Val Overall Acc: {val_metrics['overall_acc']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            best_metrics = val_metrics
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    history_path = os.path.join(experiment_dir, "training_history.csv")
    save_rows_csv(history, history_path)

    best_metrics_with_epoch = {
        **best_metrics,
        "best_val_macro_f1": best_val_f1,
        "best_epoch": max(history, key=lambda r: r["val_macro_f1"])["epoch"],
    }
    save_metrics_csv(best_metrics_with_epoch, os.path.join(experiment_dir, "best_val_metrics.csv"))

    print(f"Best Val Macro F1: {best_val_f1:.4f}")
    print(f"Saved training history to: {history_path}")

    return best_metrics, model


@torch.no_grad()
def evaluate_two_stage_full(
    keyword_model,
    data,
    config,
    device="cuda",
    vad_preds=None,
    vad_model=None,
    sampling_rate=16000,
    silence_class_idx=0,
    stage2_to_final=None,
    save_dir=None,
    split_name="test",
):
    paths, labels_final = data
    labels_final = np.asarray(labels_final, dtype=np.int64)

    if stage2_to_final is None:
        stage2_to_final = {i: i + 1 for i in range(11)}

    if vad_preds is None:
        if vad_model is None:
            vad_model = SileroVADClassifier()

        computed = []
        import torchaudio
        import torchaudio.functional as F

        for path in tqdm(paths, desc="Silero VAD eval", leave=False):
            wav, sr = torchaudio.load(path)
            if sr != sampling_rate:
                wav = F.resample(wav, sr, sampling_rate)
            if wav.dim() == 2:
                wav = wav.squeeze(0)
            computed.append(vad_model.predict(wav.cpu(), sampling_rate=sampling_rate))
        vad_preds = computed

    vad_preds = np.asarray(vad_preds, dtype=np.int64)
    if len(vad_preds) != len(paths):
        raise ValueError("Długość vad_preds musi być taka sama jak liczba próbek w data.")

    eval_dataset = SpeechCommandsDataset(
        file_paths=paths,
        labels=labels_final.tolist(),
        config=config,
        noise_paths=None,
        is_train=False,
    )

    keyword_model.eval()
    criterion = nn.CrossEntropyLoss()

    final_preds = []
    keyword_true = []
    keyword_pred = []
    keyword_losses = []

    for i, vad_pred in enumerate(tqdm(vad_preds, desc="Two-stage final eval", leave=False)):
        true_final = int(labels_final[i])

        if int(vad_pred) == 0:
            final_preds.append(silence_class_idx)
            continue

        features, _ = eval_dataset[i]
        logits = keyword_model(features.unsqueeze(0).to(device))
        pred_stage2 = int(torch.argmax(logits, dim=1).item())
        pred_final = int(stage2_to_final[pred_stage2])
        final_preds.append(pred_final)

        if true_final != silence_class_idx:
            true_stage2 = true_final - 1
            keyword_true.append(true_stage2)
            keyword_pred.append(pred_stage2)
            loss = criterion(logits, torch.tensor([true_stage2], dtype=torch.long, device=device))
            keyword_losses.append(float(loss.item()))

    final_preds = np.asarray(final_preds, dtype=np.int64)
    vad_true = (labels_final != silence_class_idx).astype(np.int64)

    results = {
        "vad": {
            **_classification_metrics(vad_true, vad_preds),
            "pred_speech": int((vad_preds == 1).sum()),
            "pred_background_noise": int((vad_preds == 0).sum()),
            "true_speech": int((vad_true == 1).sum()),
            "true_background_noise": int((vad_true == 0).sum()),
            "false_positive_background_as_speech": int(((labels_final == 0) & (vad_preds == 1)).sum()),
            "false_negative_speech_as_background": int(((labels_final != 0) & (vad_preds == 0)).sum()),
        },
        "keyword_on_vad_speech_true_nonbackground": {
            **_classification_metrics(keyword_true, keyword_pred),
            "loss": float(np.mean(keyword_losses)) if keyword_losses else np.nan,
        },
        "final_12class": _classification_metrics(labels_final, final_preds),
    }

    if save_dir is not None:
        save_metrics_csv(results, os.path.join(save_dir, f"{split_name}_metrics_two_stage_final_12class.csv"))
        save_confusion_matrix_csv(
            labels_final,
            final_preds,
            labels=list(range(12)),
            path=os.path.join(save_dir, f"{split_name}_confusion_matrix_final_12class.csv"),
        )

    return results

Writing engine.py


# data_utils.py

In [ ]:
!rm data_utils.py

In [ ]:
%%writefile data_utils.py
import os
import re
import csv
import json
import random
import hashlib
import torchaudio
import numpy as np
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

from dataset import SpeechCommandsDataset
from models import SileroVADClassifier

TARGET_WORDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]

MAX_NUM_WAVS_PER_CLASS = 2**27 - 1


def set_global_seed(config):
    seed = config.get("seed", 42)

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    if config.get("disable_tf32", True):
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

    if config.get("deterministic_algorithms", False):
        torch.use_deterministic_algorithms(
            True,
            warn_only=config.get("deterministic_warn_only", True),
        )

    return seed


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def validate_data_pipeline_config(config):
    preprocessing = config.get("preprocessing")
    augmentation = config.get("augmentation", "none")
    model_type = config.get("model_type", "").upper()

    valid_preprocessing = {"mel_spectrogram", "mfcc", "raw"}
    valid_augmentations = {"none", "speed_perturbation", "background_noise", "spec_augment"}

    if preprocessing not in valid_preprocessing:
        raise ValueError(f"Nieznany preprocessing={preprocessing}. Dozwolone: {sorted(valid_preprocessing)}")

    if augmentation not in valid_augmentations:
        raise ValueError(f"Nieznana augmentation={augmentation}. Dozwolone: {sorted(valid_augmentations)}")

    if augmentation == "spec_augment" and preprocessing != "mel_spectrogram":
        pass
        # raise ValueError("SpecAugment stosuj tylko dla preprocessing='mel_spectrogram'.")

    if preprocessing == "mfcc" and config.get("input_size") != config.get("n_mfcc", 40):
        raise ValueError(
            "Dla preprocessing='mfcc' ustaw input_size == n_mfcc, np. input_size=40."
        )

    if model_type == "AST":
        # if preprocessing != "mel_spectrogram":
        #     raise ValueError("AST wymaga preprocessing='mel_spectrogram'.")
        # if config.get("input_size", 128) != 128:
        #     raise ValueError("Dla AST wg paperu użyj input_size=128.")
        # if config.get("max_length", 100) != 100:
        #     raise ValueError("Dla Speech Commands 1s i AST wg paperu użyj max_length=100.")
        # if not config.get("ast_use_kaldi_fbank", True):
        #     raise ValueError("Dla AST wg paperu zostaw ast_use_kaldi_fbank=True.")
        pass

    if config.get("vad_filter_on_augmented", False) and augmentation == "spec_augment":
        print(
            "[WARN] vad_filter_on_augmented=True, ale spec_augment jest augmentacją cech, "
            "nie waveformu. VAD nadal zobaczy waveform bez SpecAugment."
        )

    if config.get("vad_cache_include_augmentation", True) is False:
        print(
            "[WARN] vad_cache_include_augmentation=False: różne augmentacje mogą użyć tego samego cache VAD. "
            "Zostaw True, jeśli porównujesz eksperymenty z różnymi augmentacjami."
        )


def which_set(filename, validation_percentage=10, testing_percentage=10):
    base_name = os.path.basename(filename)
    hash_name = re.sub(r"_nohash_.*$", "", base_name)
    hash_name_hashed = hashlib.sha1(hash_name.encode("utf-8")).hexdigest()

    percentage_hash = (
        (int(hash_name_hashed, 16) % (MAX_NUM_WAVS_PER_CLASS + 1))
        * (100.0 / MAX_NUM_WAVS_PER_CLASS)
    )

    if percentage_hash < validation_percentage:
        return "validation"
    elif percentage_hash < validation_percentage + testing_percentage:
        return "testing"
    else:
        return "training"


def get_noise_paths(data_path):
    noise_dir = os.path.join(data_path, "_background_noise_")
    if not os.path.exists(noise_dir):
        return []

    return [
        os.path.join(noise_dir, f)
        for f in sorted(os.listdir(noise_dir))
        if f.endswith(".wav")
    ]


def build_background_noise_chunks(
    data_path,
    sampling_rate=16000,
    clip_duration_sec=1.0,
):
    noise_paths = get_noise_paths(data_path)
    if len(noise_paths) == 0:
        return []

    clip_num_samples = int(sampling_rate * clip_duration_sec)
    cache_dir = os.path.join(
        data_path,
        f"_background_noise_chunks_{sampling_rate}_{int(clip_duration_sec * 1000)}ms",
    )
    os.makedirs(cache_dir, exist_ok=True)

    chunk_paths = []

    for noise_path in noise_paths:
        base_name = os.path.splitext(os.path.basename(noise_path))[0]

        wav, sr = torchaudio.load(noise_path)

        if sr != sampling_rate:
            wav = torchaudio.functional.resample(wav, sr, sampling_rate)

        if wav.dim() == 1:
            wav = wav.unsqueeze(0)

        if wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)

        total_num_samples = wav.shape[1]

        if total_num_samples < clip_num_samples:
            repeat_times = (clip_num_samples + total_num_samples - 1) // total_num_samples
            wav = wav.repeat(1, repeat_times)
            total_num_samples = wav.shape[1]

        num_chunks = total_num_samples // clip_num_samples

        for i in range(num_chunks):
            start = i * clip_num_samples
            end = start + clip_num_samples
            chunk = wav[:, start:end]

            chunk_name = f"{base_name}_nohash_{i}.wav"
            chunk_path = os.path.join(cache_dir, chunk_name)

            if not os.path.exists(chunk_path):
                torchaudio.save(chunk_path, chunk, sampling_rate)

            chunk_paths.append(chunk_path)

    return chunk_paths


def prepare_data_lists(
    data_path,
    target_words=TARGET_WORDS,
    approach="single_stage",
    validation_percentage=10,
    testing_percentage=10,
    seed=42,
    background_noise_sampling_rate=16000,
    background_noise_clip_duration_sec=1.0,
):
    random.seed(seed)

    splits = {
        "training": {"paths": [], "labels": [], "unknown": []},
        "validation": {"paths": [], "labels": [], "unknown": []},
        "testing": {"paths": [], "labels": [], "unknown": []},
    }

    background_noise_label = 0
    word_to_label = {word: idx + 1 for idx, word in enumerate(target_words)}
    unknown_label = len(target_words) + 1

    for folder_name in sorted(os.listdir(data_path)):
        folder_path = os.path.join(data_path, folder_name)

        if not os.path.isdir(folder_path):
            continue

        if folder_name == "_background_noise_":
            continue

        if folder_name.startswith("_background_noise_chunks_"):
            continue

        wav_files = [
            os.path.join(folder_path, f)
            for f in sorted(os.listdir(folder_path))
            if f.endswith(".wav")
        ]

        for wav_path in wav_files:
            split_name = which_set(
                wav_path,
                validation_percentage=validation_percentage,
                testing_percentage=testing_percentage,
            )

            if folder_name in target_words:
                splits[split_name]["paths"].append(wav_path)
                splits[split_name]["labels"].append(word_to_label[folder_name])
            else:
                splits[split_name]["unknown"].append(wav_path)

    for split_name in ["training", "validation", "testing"]:
        known_count = len(splits[split_name]["paths"])
        avg_target_samples = known_count // len(target_words)

        unknown_candidates = splits[split_name]["unknown"]
        sampled_unknown = random.sample(
            unknown_candidates,
            min(avg_target_samples, len(unknown_candidates)),
        )

        splits[split_name]["paths"].extend(sampled_unknown)
        splits[split_name]["labels"].extend([unknown_label] * len(sampled_unknown))

    background_noise_chunk_paths = build_background_noise_chunks(
        data_path=data_path,
        sampling_rate=background_noise_sampling_rate,
        clip_duration_sec=background_noise_clip_duration_sec,
    )

    background_noise_splits = {
        "training": [],
        "validation": [],
        "testing": [],
    }

    for chunk_path in background_noise_chunk_paths:
        split_name = which_set(
            chunk_path,
            validation_percentage=validation_percentage,
            testing_percentage=testing_percentage,
        )
        background_noise_splits[split_name].append(chunk_path)

    for split_name in ["training", "validation", "testing"]:
        known_count = len([
            y for y in splits[split_name]["labels"]
            if y in word_to_label.values()
        ])
        avg_target_samples = max(1, known_count // len(target_words))

        background_noise_candidates = background_noise_splits[split_name]
        sampled_background_noise = random.sample(
            background_noise_candidates,
            min(avg_target_samples, len(background_noise_candidates)),
        )

        splits[split_name]["paths"].extend(sampled_background_noise)
        splits[split_name]["labels"].extend([background_noise_label] * len(sampled_background_noise))

    return (
        (splits["training"]["paths"], splits["training"]["labels"]),
        (splits["validation"]["paths"], splits["validation"]["labels"]),
        (splits["testing"]["paths"], splits["testing"]["labels"]),
    )


def _copy_data(data):
    return (list(data[0]), list(data[1]))


def make_vad_cache_signature(config, noise_paths=None):
    """
    Sygnatura cache VAD.

    Domyślnie zawiera augmentację, nawet jeśli VAD filtruje raw audio.
    To celowo zapobiega przypadkowemu użyciu tego samego cache'u przy różnych eksperymentach.
    """
    signature = {
        "cache_version": 3,
        "vad_filter_on_augmented": bool(config.get("vad_filter_on_augmented", False)),
        "sample_rate": config.get("sample_rate", 16000),
        "clip_duration_sec": config.get("clip_duration_sec", 1.0),
        "vad_sampling_rate": config.get("vad_sampling_rate", 16000),
    }

    if config.get("vad_cache_include_augmentation", True):
        signature.update({
            "augmentation": config.get("augmentation", "none"),
            "speed_factors": config.get("speed_factors", [0.9, 1.0, 1.1]),
            "background_noise_prob": config.get("background_noise_prob", 0.7),
            "background_snr_min_db": config.get("background_snr_min_db", 5.0),
            "background_snr_max_db": config.get("background_snr_max_db", 20.0),
            "preprocessing": config.get("preprocessing", "mel_spectrogram"),
            "time_mask_param": config.get("time_mask_param", 8),
            "freq_mask_param": config.get("freq_mask_param", 8),
            "seed": config.get("seed", 42),
        })

    if noise_paths:
        h = hashlib.sha1()
        for p in sorted(noise_paths):
            h.update(os.path.abspath(p).encode("utf-8"))
        signature["noise_paths_hash"] = h.hexdigest()[:16]

    return signature


def _vad_cache_hash(paths, labels, sampling_rate, cache_signature=None):
    h = hashlib.sha1()
    h.update(str(sampling_rate).encode("utf-8"))

    if cache_signature is not None:
        h.update(json.dumps(cache_signature, sort_keys=True).encode("utf-8"))

    for path, label in zip(paths, labels):
        h.update(os.path.abspath(path).encode("utf-8"))
        h.update(str(label).encode("utf-8"))

    return h.hexdigest()[:16]


def _pad_or_crop_waveform(wav, num_samples):
    if wav.shape[1] > num_samples:
        return wav[:, :num_samples]
    if wav.shape[1] < num_samples:
        return torch.nn.functional.pad(wav, (0, num_samples - wav.shape[1]))
    return wav


def _load_waveform_for_vad(path, config, idx, noise_paths=None):
    sample_rate = config.get("sample_rate", 16000)
    num_samples = int(sample_rate * config.get("clip_duration_sec", 1.0))

    wav, sr = torchaudio.load(path)

    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sr, sample_rate)

    if wav.dim() == 1:
        wav = wav.unsqueeze(0)

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    aug_type = config.get("augmentation", "none")
    rng = random.Random(config.get("seed", 42) + idx * 1000003)

    if config.get("vad_filter_on_augmented", False):
        if aug_type == "speed_perturbation":
            speed_choices = config.get("speed_factors", [0.9, 1.0, 1.1])
            speed_factor = rng.choice(speed_choices)
            if speed_factor != 1.0:
                new_sr = int(sample_rate * speed_factor)
                wav = torchaudio.functional.resample(wav, sample_rate, new_sr)

        elif aug_type == "background_noise" and noise_paths:
            p = config.get("background_noise_prob", 0.7)
            if rng.random() <= p:
                noise_file = rng.choice(noise_paths)
                noise, nsr = torchaudio.load(noise_file)

                if nsr != sample_rate:
                    noise = torchaudio.functional.resample(noise, nsr, sample_rate)

                if noise.dim() == 1:
                    noise = noise.unsqueeze(0)

                if noise.shape[0] > 1:
                    noise = noise.mean(dim=0, keepdim=True)

                if noise.shape[1] > num_samples:
                    start = rng.randint(0, noise.shape[1] - num_samples)
                    noise = noise[:, start:start + num_samples]
                else:
                    noise = _pad_or_crop_waveform(noise, num_samples)

                wav_for_power = _pad_or_crop_waveform(wav, num_samples)
                speech_power = wav_for_power.norm(p=2)
                noise_power = noise.norm(p=2)

                if noise_power != 0:
                    snr_db = rng.uniform(
                        config.get("background_snr_min_db", 5.0),
                        config.get("background_snr_max_db", 20.0),
                    )
                    snr = 10 ** (snr_db / 20.0)
                    scale = speech_power / (snr * noise_power)
                    wav = wav_for_power + scale * noise

    wav = _pad_or_crop_waveform(wav, num_samples)

    vad_sr = config.get("vad_sampling_rate", sample_rate)
    if vad_sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sample_rate, vad_sr)

    return wav.squeeze(0)


def get_vad_predictions_cached(
    data,
    split_name,
    cache_dir,
    sampling_rate=16000,
    batch_size=64,
    force_recompute=False,
    cache_signature=None,
    config=None,
    noise_paths=None,
):
    paths, labels = data
    os.makedirs(cache_dir, exist_ok=True)

    cache_hash = _vad_cache_hash(paths, labels, sampling_rate, cache_signature)
    cache_path = os.path.join(
        cache_dir,
        f"silero_vad_{split_name}_{sampling_rate}_{cache_hash}.csv",
    )

    if os.path.exists(cache_path) and not force_recompute:
        cached_paths = []
        cached_preds = []

        with open(cache_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                cached_paths.append(row["path"])
                cached_preds.append(int(row["vad_pred"]))

        if cached_paths == paths and len(cached_preds) == len(paths):
            print(f"[Silero VAD cache] loaded {split_name}: {cache_path}")
            return cached_preds

        print(f"[Silero VAD cache] cache mismatch, recomputing: {cache_path}")

    vad = SileroVADClassifier()
    preds = []

    for start in tqdm(range(0, len(paths), batch_size), desc=f"Silero VAD {split_name}", leave=False):
        batch_paths = paths[start:start + batch_size]
        batch_waveforms = []

        for local_i, path in enumerate(batch_paths):
            global_i = start + local_i

            if config is None:
                wav, sr = torchaudio.load(path)
                if sr != sampling_rate:
                    wav = torchaudio.functional.resample(wav, sr, sampling_rate)
                if wav.dim() == 2:
                    wav = wav.squeeze(0)
            else:
                wav = _load_waveform_for_vad(path, config, global_i, noise_paths=noise_paths)

            batch_waveforms.append(wav)

        batch_preds = vad.predict_batch(batch_waveforms, sampling_rate=sampling_rate)
        preds.extend(batch_preds.tolist())

    with open(cache_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["path", "label", "vad_pred", "cache_signature_json"],
        )
        writer.writeheader()
        signature_json = json.dumps(cache_signature, sort_keys=True, ensure_ascii=False) if cache_signature else ""
        for path, label, pred in zip(paths, labels, preds):
            writer.writerow({
                "path": path,
                "label": label,
                "vad_pred": int(pred),
                "cache_signature_json": signature_json,
            })

    print(f"[Silero VAD cache] saved {split_name}: {cache_path}")
    return preds


def _summarize_vad(labels, vad_preds):
    labels = np.asarray(labels)
    vad_preds = np.asarray(vad_preds)

    stats = {
        "pred_speech": int((vad_preds == 1).sum()),
        "pred_background_noise": int((vad_preds == 0).sum()),
        "true_speech": int((labels != 0).sum()),
        "true_background_noise": int((labels == 0).sum()),
        "false_positive_background_as_speech": int(((labels == 0) & (vad_preds == 1)).sum()),
        "false_negative_speech_as_background": int(((labels != 0) & (vad_preds == 0)).sum()),
        "true_positive_speech": int(((labels != 0) & (vad_preds == 1)).sum()),
        "true_negative_background": int(((labels == 0) & (vad_preds == 0)).sum()),
    }
    return stats


def _print_vad_stats(name, stats):
    print(f"\n[Silero VAD] {name}")
    print(f"pred speech:                 {stats['pred_speech']}")
    print(f"pred background_noise:       {stats['pred_background_noise']}")
    print(f"true speech:                 {stats['true_speech']}")
    print(f"true background_noise:       {stats['true_background_noise']}")
    print(f"FP background -> speech:     {stats['false_positive_background_as_speech']}")
    print(f"FN speech -> background:     {stats['false_negative_speech_as_background']}")


def _make_stage2_data_from_vad(data, vad_preds=None):
    paths, labels = data

    kept_paths, kept_labels = [], []
    removed_by_vad_paths, removed_by_vad_labels = [], []
    removed_background_fp_paths, removed_background_fp_labels = [], []

    if vad_preds is None:
        vad_preds = [1] * len(paths)

    for path, label, vad_pred in zip(paths, labels, vad_preds):
        if int(vad_pred) != 1:
            removed_by_vad_paths.append(path)
            removed_by_vad_labels.append(label)
            continue

        if int(label) == 0:
            # background_noise nigdy nie trafia do 11-klasowego etapu 2.
            removed_background_fp_paths.append(path)
            removed_background_fp_labels.append(label)
            continue

        kept_paths.append(path)
        kept_labels.append(int(label) - 1)

    if len(kept_labels) > 0:
        assert min(kept_labels) >= 0 and max(kept_labels) <= 10, (
            "Stage2 labels muszą być w zakresie 0..10."
        )

    return (
        (kept_paths, kept_labels),
        (removed_by_vad_paths, removed_by_vad_labels),
        (removed_background_fp_paths, removed_background_fp_labels),
    )


def build_loaders(config, return_data=False):
    validate_data_pipeline_config(config)

    data_path = config["data_path"]
    noise_paths = get_noise_paths(data_path)
    vad_cache_signature = make_vad_cache_signature(config, noise_paths=noise_paths)

    train_data, val_data, test_data = prepare_data_lists(
        data_path=data_path,
        target_words=config["target_words"],
        approach=config["approach"],
        validation_percentage=config.get("validation_percentage", 10),
        testing_percentage=config.get("testing_percentage", 10),
        seed=config.get("seed", 42),
        background_noise_sampling_rate=config.get("background_noise_sampling_rate", 16000),
        background_noise_clip_duration_sec=config.get("background_noise_clip_duration_sec", 1.0),
    )

    train_full_data = _copy_data(train_data)
    val_full_data = _copy_data(val_data)
    test_full_data = _copy_data(test_data)

    print("Unique train labels before stage handling:", sorted(set(train_data[1])))
    print("Unique val labels before stage handling:", sorted(set(val_data[1])))
    print("Unique test labels before stage handling:", sorted(set(test_data[1])))

    metadata = {
        "train_full_data": train_full_data,
        "val_full_data": val_full_data,
        "test_full_data": test_full_data,
        "stage2_to_final": {i: i + 1 for i in range(11)},
        "final_label_names": ["background_noise"] + list(config["target_words"]) + ["unknown"],
        "stage2_label_names": list(config["target_words"]) + ["unknown"],
        "vad_cache_signature": vad_cache_signature,
    }

    if config["approach"] == "two_stage":
        cache_dir = config.get(
            "vad_cache_dir",
            os.path.join(data_path, "_cache", "silero_vad"),
        )

        if config.get("use_silero_vad_filtering", True):
            train_vad_preds = get_vad_predictions_cached(
                train_data,
                split_name="train",
                cache_dir=cache_dir,
                sampling_rate=config.get("vad_sampling_rate", 16000),
                batch_size=config.get("vad_batch_size", 64),
                force_recompute=config.get("force_vad_recompute", False),
                cache_signature=vad_cache_signature,
                config=config,
                noise_paths=noise_paths,
            )
            val_vad_preds = get_vad_predictions_cached(
                val_data,
                split_name="val",
                cache_dir=cache_dir,
                sampling_rate=config.get("vad_sampling_rate", 16000),
                batch_size=config.get("vad_batch_size", 64),
                force_recompute=config.get("force_vad_recompute", False),
                cache_signature=vad_cache_signature,
                config=config,
                noise_paths=noise_paths,
            )
            test_vad_preds = get_vad_predictions_cached(
                test_data,
                split_name="test",
                cache_dir=cache_dir,
                sampling_rate=config.get("vad_sampling_rate", 16000),
                batch_size=config.get("vad_batch_size", 64),
                force_recompute=config.get("force_vad_recompute", False),
                cache_signature=vad_cache_signature,
                config=config,
                noise_paths=noise_paths,
            )
        else:
            train_vad_preds = [1] * len(train_data[0])
            val_vad_preds = [1] * len(val_data[0])
            test_vad_preds = [1] * len(test_data[0])

        train_vad_stats = _summarize_vad(train_data[1], train_vad_preds)
        val_vad_stats = _summarize_vad(val_data[1], val_vad_preds)
        test_vad_stats = _summarize_vad(test_data[1], test_vad_preds)

        _print_vad_stats("TRAIN", train_vad_stats)
        _print_vad_stats("VAL", val_vad_stats)
        _print_vad_stats("TEST", test_vad_stats)

        train_data, train_removed_by_vad, train_removed_background_fp = _make_stage2_data_from_vad(train_data, train_vad_preds)
        val_data, val_removed_by_vad, val_removed_background_fp = _make_stage2_data_from_vad(val_data, val_vad_preds)
        test_data, test_removed_by_vad, test_removed_background_fp = _make_stage2_data_from_vad(test_data, test_vad_preds)

        metadata.update({
            "train_vad_preds": train_vad_preds,
            "val_vad_preds": val_vad_preds,
            "test_vad_preds": test_vad_preds,
            "train_vad_stats": train_vad_stats,
            "val_vad_stats": val_vad_stats,
            "test_vad_stats": test_vad_stats,
            "vad_cache_dir": cache_dir,
            "train_stage2_data": train_data,
            "val_stage2_data": val_data,
            "test_stage2_data": test_data,
            "train_removed_by_vad": train_removed_by_vad,
            "val_removed_by_vad": val_removed_by_vad,
            "test_removed_by_vad": test_removed_by_vad,
            "train_removed_background_fp": train_removed_background_fp,
            "val_removed_background_fp": val_removed_background_fp,
            "test_removed_background_fp": test_removed_background_fp,
        })

        print(f"\n[Two-stage] train kept for stage2: {len(train_data[0])}")
        print(f"[Two-stage] train removed by VAD: {len(train_removed_by_vad[0])}")
        print(f"[Two-stage] train removed background FP: {len(train_removed_background_fp[0])}")

        print(f"[Two-stage] val kept for stage2:   {len(val_data[0])}")
        print(f"[Two-stage] val removed by VAD:   {len(val_removed_by_vad[0])}")
        print(f"[Two-stage] val removed background FP: {len(val_removed_background_fp[0])}")

        print(f"[Two-stage] test kept for stage2:  {len(test_data[0])}")
        print(f"[Two-stage] test removed by VAD:  {len(test_removed_by_vad[0])}")
        print(f"[Two-stage] test removed background FP: {len(test_removed_background_fp[0])}")

    print("Final train labels used by trained model:", sorted(set(train_data[1])))
    print("Final val labels used by trained model:", sorted(set(val_data[1])))
    print("Final test labels used by trained model:", sorted(set(test_data[1])))

    train_dataset = SpeechCommandsDataset(
        file_paths=train_data[0],
        labels=train_data[1],
        config=config,
        noise_paths=noise_paths,
        is_train=True,
    )

    val_dataset = SpeechCommandsDataset(
        file_paths=val_data[0],
        labels=val_data[1],
        config=config,
        noise_paths=None,
        is_train=False,
    )

    test_dataset = SpeechCommandsDataset(
        file_paths=test_data[0],
        labels=test_data[1],
        config=config,
        noise_paths=None,
        is_train=False,
    )

    g = torch.Generator()
    g.manual_seed(config.get("seed", 42))

    common_loader_kwargs = dict(
        num_workers=config.get("num_workers", 2),
        pin_memory=config.get("pin_memory", True),
        worker_init_fn=seed_worker,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.get("batch_size", 64),
        shuffle=True,
        generator=g,
        **common_loader_kwargs,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.get("batch_size", 64),
        shuffle=False,
        **common_loader_kwargs,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=config.get("batch_size", 64),
        shuffle=False,
        **common_loader_kwargs,
    )

    if return_data:
        return train_loader, val_loader, test_loader, metadata

    return train_loader, val_loader, test_loader

Writing data_utils.py


# END OF .py FILES

In [ ]:
%%writefile run_experiments.py
import os
import copy
import torch

from data_utils import build_loaders, TARGET_WORDS, set_global_seed
from engine import (
    run_experiment,
    evaluate_model,
    evaluate_two_stage_full,
    save_metrics_csv,
    save_confusion_matrix_csv,
    save_json,
)


BASE_CONFIG = {
    "data_path": "/content/tf_speech/train/audio",
    "target_words": TARGET_WORDS,
    "validation_percentage": 10,
    "testing_percentage": 10,

    # VAD/cache
    "use_silero_vad_filtering": True,
    "vad_sampling_rate": 16000,
    "vad_batch_size": 64,
    "force_vad_recompute": False,
    "vad_cache_include_augmentation": True,
    "vad_filter_on_augmented": True,

    # augmentation params
    "speed_factors": [0.9, 1.0, 1.1],
    "background_noise_prob": 0.7,
    "background_snr_min_db": 5.0,
    "background_snr_max_db": 20.0,
    "specaugment_prob": 0.8,
    "num_time_masks": 2,
    "time_mask_param": 8,

    "frequency_stride": 10,
    "time_stride": 10,
    "num_freq_masks": 2,
    "freq_mask_param": 8,

    # training
    "num_workers": 24,
    "num_epochs": 15,
    "save_dir": "drive/MyDrive/checkpoints/single_stage",
    "batch_size": 64,

    # determinism
    "deterministic_algorithms": False,
    "deterministic_warn_only": True,
    "disable_tf32": True,
}


# EXPERIMENTS = [
#     # {
#     #     "experiment_name": "two_stage_lstm_mel_background_noise_seed67",
#     #     "approach": "two_stage",
#     #     "model_type": "LSTM",
#     #     "num_classes": 11,
#     #     "num_layers": 2,
#     #     "dropout": 0.1,
#     #     "learning_rate": 1e-3,
#     #     "augmentation": "background_noise",
#     #     "background_noise_prob": 0.7,
#     #     "background_snr_min_db": 5.0,
#     #     "background_snr_max_db": 20.0,
#     #     # ustaw True tylko jeśli naprawdę chcesz, żeby VAD-filter stage2 zależał od waveform augmentacji
#     #     "vad_filter_on_augmented": False,
#     # },
#     # {
#     #     "experiment_name": "two_stage_ast_mfcc_spec_seed_67",
#     #     "approach": "two_stage",
#     #     "model_type": "AST",
#     #     "seed": 67,
#     #     "num_classes": 11,

#     #     "preprocessing": "mfcc",
#     #     "n_mfcc": 40,
#     #     "mfcc_n_mels": 128,

#     #     # dla AST input_size powinien odpowiadać liczbie cech na ramkę
#     #     "input_size": 40,
#     #     "max_length": 100,

#     #     "num_heads": 12,
#     #     "hidden_size": 768,
#     #     "num_hidden_layers": 12,
#     #     "patch_size": 16,
#     #     "frequency_stride": 10,
#     #     "time_stride": 10,

#     #     "ast_normalize": True,
#     #     "augmentation": "spec_augment",

#     #     "learning_rate": 1e-5,
#     #     "num_epochs": 15,
#     #     "batch_size": 64,
#     #     "use_ast_paper_lr_decay": False,
#     # },
#      {
#         "experiment_name": "two_stage_ast_mfcc_spec_seed_167",
#         "approach": "two_stage",
#         "model_type": "AST",
#         "seed": 167,
#         "num_classes": 11,

#         "preprocessing": "mfcc",
#         "n_mfcc": 40,
#         "mfcc_n_mels": 128,

#         # dla AST input_size powinien odpowiadać liczbie cech na ramkę
#         "input_size": 40,
#         "max_length": 100,

#         "num_heads": 12,
#         "hidden_size": 768,
#         "num_hidden_layers": 12,
#         "patch_size": 16,
#         "frequency_stride": 10,
#         "time_stride": 10,

#         "ast_normalize": True,
#         "augmentation": "spec_augment",

#         "learning_rate": 1e-5,
#         "num_epochs": 15,
#         "batch_size": 64,
#         "use_ast_paper_lr_decay": False,
#     },
#      {
#         "experiment_name": "two_stage_ast_mfcc_spec_seed_267",
#         "approach": "two_stage",
#         "model_type": "AST",
#         "seed": 267,
#         "num_classes": 11,

#         "preprocessing": "mfcc",
#         "n_mfcc": 40,
#         "mfcc_n_mels": 128,

#         # dla AST input_size powinien odpowiadać liczbie cech na ramkę
#         "input_size": 40,
#         "max_length": 100,

#         "num_heads": 12,
#         "hidden_size": 768,
#         "num_hidden_layers": 12,
#         "patch_size": 16,
#         "frequency_stride": 10,
#         "time_stride": 10,

#         "ast_normalize": True,
#         "augmentation": "spec_augment",

#         "learning_rate": 1e-5,
#         "num_epochs": 15,
#         "batch_size": 64,
#         "use_ast_paper_lr_decay": False,
#     },
# ]

# AST_NUM_HEADS = [4, 8]
SEEDS = [67, 167, 267]
# DROPOUTS = [0.5]
# LEARNING_RATE = [1e-4]
# COSINE_ANNEALING = [True]

BASE_AST_EXPERIMENT = {
    "approach": "single_stage",
    "model_type": "AST",
    "num_classes": 12,

    "preprocessing": "mfcc",
    "n_mfcc": 40,
    "mfcc_n_mels": 128,

    "input_size": 40,
    "max_length": 100,
    "num_heads": 12,

    "hidden_size": 768,
    "num_hidden_layers": 12,
    "patch_size": 16,
    "frequency_stride": 10,
    "time_stride": 10,

    "ast_normalize": True,
    "augmentation": "none",
    "num_epochs": 15,
    "batch_size": 64,
    "use_cosine_annealing": True,
    "learning_rate": 1e-4,
    "use_ast_paper_lr_decay": False,
}


EXPERIMENTS = [
    {
        **BASE_AST_EXPERIMENT,
        "seed": seed,
        "dropout": 0.1,
        "experiment_name": (
            f"single_stage_ast_mfcc_none_seed_{seed}"
        ),
    }
    for seed in SEEDS
]
print(EXPERIMENTS)

def make_config(base_config, overrides):
    config = copy.deepcopy(base_config)
    config.update(overrides)

    if "experiment_name" not in config:
        config["experiment_name"] = (
            f"{config['approach']}_{config['model_type']}_"
            f"{config['preprocessing']}_{config.get('augmentation', 'none')}_"
            f"seed{config.get('seed', 42)}"
        )

    return config


def main():
    for exp in EXPERIMENTS:
        config = make_config(BASE_CONFIG, exp)
        set_global_seed(config)

        device = "cuda" if torch.cuda.is_available() else "cpu"

        experiment_dir = os.path.join(config["save_dir"], config["experiment_name"])
        os.makedirs(experiment_dir, exist_ok=True)
        save_json(config, os.path.join(experiment_dir, "config.json"))

        print("\n" + "#" * 100)
        print(f"Running experiment: {config['experiment_name']}")
        print("#" * 100)
        print(config)
        train_loader, val_loader, test_loader, metadata = build_loaders(
            config,
            return_data=True,
        )

        save_json(metadata["vad_cache_signature"], os.path.join(experiment_dir, "vad_cache_signature.json"))

        if config["approach"] == "two_stage":
            save_metrics_csv(metadata["train_vad_stats"], os.path.join(experiment_dir, "train_vad_stats.csv"))
            save_metrics_csv(metadata["val_vad_stats"], os.path.join(experiment_dir, "val_vad_stats.csv"))
            save_metrics_csv(metadata["test_vad_stats"], os.path.join(experiment_dir, "test_vad_stats.csv"))

        best_metrics, model = run_experiment(
            config=config,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=config["num_epochs"],
            device=device,
        )

        checkpoint_path = os.path.join(experiment_dir, "best_model.pth")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": config,
                "best_val_metrics": best_metrics,
            },
            checkpoint_path,
        )
        print(f"Saved model to: {checkpoint_path}")

        criterion = torch.nn.CrossEntropyLoss()
        model_space_test_metrics, y_true_model, y_pred_model = evaluate_model(
            model,
            test_loader,
            criterion,
            device,
            return_preds=True,
        )
        save_metrics_csv(
            model_space_test_metrics,
            os.path.join(experiment_dir, "test_metrics_model_space.csv"),
        )
        save_confusion_matrix_csv(
            y_true_model,
            y_pred_model,
            labels=list(range(config["num_classes"])),
            path=os.path.join(experiment_dir, "test_confusion_matrix_model_space.csv"),
        )

        print("Test metrics in model label space:")
        for k, v in model_space_test_metrics.items():
            if isinstance(v, float):
                print(f"{k}: {v:.4f}")
            else:
                print(f"{k}: {v}")

        if config["approach"] == "two_stage":
            val_two_stage_metrics = evaluate_two_stage_full(
                keyword_model=model,
                data=metadata["val_full_data"],
                config=config,
                device=device,
                vad_preds=metadata["val_vad_preds"],
                sampling_rate=config.get("vad_sampling_rate", 16000),
                stage2_to_final=metadata["stage2_to_final"],
                save_dir=experiment_dir,
                split_name="val",
            )
            test_two_stage_metrics = evaluate_two_stage_full(
                keyword_model=model,
                data=metadata["test_full_data"],
                config=config,
                device=device,
                vad_preds=metadata["test_vad_preds"],
                sampling_rate=config.get("vad_sampling_rate", 16000),
                stage2_to_final=metadata["stage2_to_final"],
                save_dir=experiment_dir,
                split_name="test",
            )

            print("Final two-stage 12-class test metrics:")
            for section, metrics in test_two_stage_metrics.items():
                print(f"[{section}]")
                for k, v in metrics.items():
                    if isinstance(v, float):
                        print(f"  {k}: {v:.4f}")
                    else:
                        print(f"  {k}: {v}")

        print(f"Finished: {config['experiment_name']}")


if __name__ == "__main__":
    main()

Overwriting run_experiments.py


In [ ]:
# ==========================
# RUN EXPERIMENTS - Kaggle
# ==========================
import os
import torch
import importlib

import dataset
importlib.reload(dataset)

import models
importlib.reload(models)

import data_utils
importlib.reload(data_utils)

import engine
importlib.reload(engine)

from data_utils import build_loaders, TARGET_WORDS, set_global_seed
from engine import (
    run_experiment,
    evaluate_model,
    evaluate_two_stage_full,
    save_metrics_csv,
    save_confusion_matrix_csv,
    save_json,
)

import run_experiments
importlib.reload(run_experiments)
import time, subprocess


run_experiments.main()

[{'approach': 'single_stage', 'model_type': 'AST', 'num_classes': 12, 'preprocessing': 'mfcc', 'n_mfcc': 40, 'mfcc_n_mels': 128, 'input_size': 40, 'max_length': 100, 'num_heads': 12, 'hidden_size': 768, 'num_hidden_layers': 12, 'patch_size': 16, 'frequency_stride': 10, 'time_stride': 10, 'ast_normalize': True, 'augmentation': 'none', 'num_epochs': 15, 'batch_size': 64, 'use_cosine_annealing': True, 'learning_rate': 0.0001, 'use_ast_paper_lr_decay': False, 'seed': 67, 'dropout': 0.1, 'experiment_name': 'single_stage_ast_mfcc_none_seed_67'}, {'approach': 'single_stage', 'model_type': 'AST', 'num_classes': 12, 'preprocessing': 'mfcc', 'n_mfcc': 40, 'mfcc_n_mels': 128, 'input_size': 40, 'max_length': 100, 'num_heads': 12, 'hidden_size': 768, 'num_hidden_layers': 12, 'patch_size': 16, 'frequency_stride': 10, 'time_stride': 10, 'ast_normalize': True, 'augmentation': 'none', 'num_epochs': 15, 'batch_size': 64, 'use_cosine_annealing': True, 'learning_rate': 0.0001, 'use_ast_paper_lr_decay': Fa

/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                            | Status     |                                                                                               
-------------------------------+------------+-----------------------------------------------------------------------------------------------
classifier.dense.bias          | UNEXPECTED |                                                                                               
classifier.layernorm.bias      | UNEXPECTED |                                                                                               
classifier.layernorm.weight    | UNEXPECTED |                                                                                               
classifier.dense.weight        | UNEXPECTED |                                                                                               
embeddings.position_embeddings | MISMATCH   | Reinit due to size mismatch ckpt: torch.S

AST position embeddings: torch.Size([1, 29, 768])
Experiment : single_stage_ast_mfcc_none_seed_67
Approach   : single_stage
Model      : AST
Num classes: 12
LR         : 0.0001


...

Epoch 13/15 | Train Loss: 0.0469 | Val Loss: 0.4427 | Val Macro F1: 0.8955 | Val Balanced Acc: 0.8894 | Val Overall Acc: 0.9170


Epoch 14/15 | Train Loss: 0.0430 | Val Loss: 0.4574 | Val Macro F1: 0.8914 | Val Balanced Acc: 0.8869 | Val Overall Acc: 0.9184


Epoch 15/15 | Train Loss: 0.0405 | Val Loss: 0.4541 | Val Macro F1: 0.8946 | Val Balanced Acc: 0.8895 | Val Overall Acc: 0.9194
Best Val Macro F1: 0.8955
Saved training history to: drive/MyDrive/checkpoints/single_stage/single_stage_ast_mfcc_none_seed_267/training_history.csv
Saved model to: drive/MyDrive/checkpoints/single_stage/single_stage_ast_mfcc_none_seed_267/best_model.pth


Test metrics in model label space:
loss: 0.2637
macro_f1: 0.8651
balanced_acc: 0.9435
overall_acc: 0.9437
count: 2823
Finished: single_stage_ast_mfcc_none_seed_267


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [ ]:
!zip -r two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67.zip drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67

updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/ (stored 0%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/train_vad_stats.csv (deflated 48%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/val_metrics_two_stage_final_12class.csv (deflated 61%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/training_history.csv (deflated 52%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/val_vad_stats.csv (deflated 49%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/best_model.pth (deflated 7%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/test_metrics_two_stage_final_12class.csv (deflated 63%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_67/config.json (deflated 5

In [ ]:
!zip -r two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167.zip drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167

  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/ (stored 0%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/train_vad_stats.csv (deflated 48%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/val_metrics_two_stage_final_12class.csv (deflated 61%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/training_history.csv (deflated 53%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/val_vad_stats.csv (deflated 49%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/best_model.pth (deflated 7%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/test_metrics_two_stage_final_12class.csv (deflated 62%)
  adding: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_167/config.json (de

In [ ]:
!zip -r two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267.zip drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267

updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/ (stored 0%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/train_vad_stats.csv (deflated 48%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/val_metrics_two_stage_final_12class.csv (deflated 61%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/training_history.csv (deflated 52%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/val_vad_stats.csv (deflated 48%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/best_model.pth (deflated 7%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/test_metrics_two_stage_final_12class.csv (deflated 62%)
updating: drive2/MyDrive/checkpoints/lr/two_stage_ast_mfcc_none_lr_0.0001_ca_True_seed_267/config.json (de